# ============================================================
# MPATH DTC DISPLAY - Data Quality Management (DQM) Pipeline
# ============================================================
# Checks: Schema | Data Types | Null Violations
# FSD Source: mpath_us_all_brands_dtc_monthly_aggregate_dqm_schema_display.xlsx
# Dataset   : MPATH_DTC_DISPLAY
# ============================================================


In [2]:
#import libraries
import pandas as pd
import numpy as np
from datetime import datetime
import re

# ============================================================
# SECTION 1: FSD SCHEMA DEFINITION
# Derived from: mpath_us_all_brands_dtc_monthly_aggregate_dqm_schema_display.xlsx
# ============================================================

In [3]:

FSD_SCHEMA = [
    # (column_name,                         fsd_dtype,          is_not_null)
    ("AGENCY_VENDOR",                        "String(50)",        True),
    ("PLATFORM",                             "String(50)",        True),
    ("BRAND",                                "String(50)",        True),
    ("INDICATION",                           "String(60)",        True),
    ("PROFILE",                              "String(50)",        True),
    ("DATE",                                 "String(mm/yyyy)",   True),
    ("CAMPAIGN_NAME",                        "String(250)",       True),
    ("SITE",                                 "String(250)",       True),
    ("PACKAGE_ROADBLOCK",                    "String(600)",       True),
    ("PACKAGE_ROADBLOCK_ID",                 "Integer(30)",       True),
    ("PLACEMENT_NAME",                       "String(600)",       True),
    ("PLACEMENT_ID",                         "Integer",           True),
    ("CREATIVE_NAME",                        "String(500)",       True),
    ("CREATIVE_SIZE",                        "String(30)",        True),
    ("CREATIVE_TYPE",                        "String(250)",       True),
    ("BROWSER_PLATFORM",                     "String(100)",       True),
    ("DEVICE_TYPE",                          "String(100)",       True),
    ("ACTIVITY_GROUP",                       "String(300)",       True),
    ("ACTIVITY",                             "String(300)",       True),
    ("IMPRESSIONS",                          "Integer",           True),
    ("CLICKS",                               "Integer",           True),
    ("INTERACTIVE_IMPRESSION",               "Integer",           True),
    ("RICH_MEDIA_IMPRESSION",                "Integer",           True),
    ("CLICK_THROUGH_CONVERSIONS",            "Integer",           True),
    ("VIEW_THROUGH_CONVERSIONS",             "Integer",           True),
    ("TOTAL_CONVERSIONS",                    "Integer",           True),
    ("CLICK_THROUGH_ENGAGEMENTS",            "Integer",           True),
    ("VIEW_THROUGH_ENGAGEMENTS",             "Integer",           True),
    ("CLICK_THROUGH_LANDING_PAGE_CONVERSIONS","Integer",          True),
    ("VIEW_THROUGH_LANDING_PAGE_CONVERSIONS", "Integer",          True),
    ("SITE_TYPE",                            "String(25)",        True),
    ("TACTIC",                               "String(50)",        True),
    ("BANNER_TYPE",                          "Unknown",           True),  # NaN in FSD — flagged as Unknown
]

FSD_DF = pd.DataFrame(FSD_SCHEMA, columns=["column_name", "fsd_dtype", "is_not_null"])
FSD_DF.head()

,column_name,fsd_dtype,is_not_null
0,AGENCY_VENDOR,String(50),True
1,PLATFORM,String(50),True
2,BRAND,String(50),True
3,INDICATION,String(60),True
4,PROFILE,String(50),True


# ============================================================
# SECTION 2: DTYPE MAPPING UTILITY
# Maps FSD dtype labels → expected pandas dtype families
# ============================================================

In [4]:
def resolve_expected_pandas_dtype(fsd_dtype: str) -> str:
    """
    Maps an FSD data type string to its expected pandas dtype category.

    Returns one of: 'string', 'integer', 'float', 'datetime', 'unknown'
    """
    if pd.isna(fsd_dtype) or fsd_dtype.strip().lower() == "unknown":
        return "unknown"

    fsd_lower = fsd_dtype.strip().lower()

    if fsd_lower.startswith("string"):
        return "string"
    elif fsd_lower.startswith("integer"):
        return "integer"
    elif fsd_lower.startswith("float") or fsd_lower.startswith("decimal"):
        return "float"
    elif fsd_lower.startswith("date") or "datetime" in fsd_lower:
        return "datetime"
    else:
        return "unknown"

In [5]:
def get_actual_pandas_dtype_category(series: pd.Series) -> str:
    """
    Categorizes a pandas Series into a generic dtype family:
    'string', 'integer', 'float', 'datetime', or 'other'.
    """
    dtype = series.dtype

    if pd.api.types.is_integer_dtype(dtype):
        return "integer"
    elif pd.api.types.is_float_dtype(dtype):
        return "float"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "datetime"
    elif pd.api.types.is_object_dtype(dtype) or pd.api.types.is_string_dtype(dtype):
        return "string"
    else:
        return "other"

# ============================================================
# SECTION 3: DQM CHECK FUNCTIONS
# ============================================================

In [ ]:
# ------------------------------------------------------------
# CHECK 1: Schema Check
# Validates that all FSD columns are present in the dataset
# and flags any extra columns not defined in the FSD.
# ------------------------------------------------------------

def check_schema(df: pd.DataFrame, fsd_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compares FSD column definitions against the actual dataset columns.

    Anomaly types detected:
      - MISSING_COLUMN   : Column defined in FSD but absent from dataset
      - EXTRA_COLUMN     : Column present in dataset but not defined in FSD
    """
    anomalies = []
    dataset_cols = set(df.columns.str.strip().str.upper())
    fsd_cols     = set(fsd_df["column_name"].str.strip().str.upper())

    # Columns in FSD but missing from dataset
    for col in sorted(fsd_cols - dataset_cols):
        anomalies.append({
            "check_type"       : "SCHEMA",
            "column_name"      : col,
            "anomaly_type"     : "MISSING_COLUMN",
            "fsd_expected"     : "Column must exist",
            "actual_value"     : "Column not found in dataset",
            "affected_row_count": None,
            "details"          : f"Column '{col}' is defined in FSD but missing from the dataset.",
        })

    # Columns in dataset but not in FSD
    for col in sorted(dataset_cols - fsd_cols):
        anomalies.append({
            "check_type"       : "SCHEMA",
            "column_name"      : col,
            "anomaly_type"     : "EXTRA_COLUMN",
            "fsd_expected"     : "Column not defined in FSD",
            "actual_value"     : "Column exists in dataset",
            "affected_row_count": None,
            "details"          : f"Column '{col}' exists in the dataset but is NOT defined in the FSD.",
        })

    return pd.DataFrame(anomalies)


In [ ]:
# ------------------------------------------------------------
# CHECK 2: Data Type Check
# Validates that each column's dtype in the dataset matches
# the expected dtype family defined in the FSD.
# ------------------------------------------------------------

def check_datatypes(df: pd.DataFrame, fsd_df: pd.DataFrame) -> pd.DataFrame:
    """
    For every column present in BOTH the FSD and the dataset,
    compares the expected FSD dtype family vs. the actual pandas dtype.

    Anomaly types detected:
      - DTYPE_MISMATCH : Actual dtype does not match FSD-specified dtype family
    """
    anomalies   = []
    dataset_cols_upper = {c.strip().upper(): c for c in df.columns}

    for _, row in fsd_df.iterrows():
        col_name    = row["column_name"].strip().upper()
        fsd_dtype   = row["fsd_dtype"]

        # Skip columns absent from dataset (caught by schema check)
        if col_name not in dataset_cols_upper:
            continue

        actual_col_name    = dataset_cols_upper[col_name]
        expected_category  = resolve_expected_pandas_dtype(fsd_dtype)
        actual_category    = get_actual_pandas_dtype_category(df[actual_col_name])
        actual_pandas_dtype = str(df[actual_col_name].dtype)

        # Skip if FSD dtype is unknown (no basis for comparison)
        if expected_category == "unknown":
            continue

        if expected_category != actual_category:
            anomalies.append({
                "check_type"       : "DTYPE",
                "column_name"      : col_name,
                "anomaly_type"     : "DTYPE_MISMATCH",
                "fsd_expected"     : f"{fsd_dtype}  →  [{expected_category}]",
                "actual_value"     : f"{actual_pandas_dtype}  →  [{actual_category}]",
                "affected_row_count": len(df),
                "details"          : (
                    f"Column '{col_name}': FSD expects dtype family "
                    f"'{expected_category}' (FSD: '{fsd_dtype}'), "
                    f"but actual pandas dtype is '{actual_pandas_dtype}' "
                    f"(family: '{actual_category}')."
                ),
            })

    return pd.DataFrame(anomalies)


In [ ]:
# ------------------------------------------------------------
# CHECK 3: Null Value Check
# Flags columns marked as NOT NULL in FSD that contain nulls.
# ------------------------------------------------------------


def check_nulls(df: pd.DataFrame, fsd_df: pd.DataFrame) -> pd.DataFrame:
    """
    For every column flagged as NOT NULL in the FSD (Is Not Null? == 'Y'),
    checks whether any null values exist in the actual dataset.

    Anomaly types detected:
      - NULL_VIOLATION : NOT NULL column contains null/NaN values
    """
    anomalies   = []
    dataset_cols_upper = {c.strip().upper(): c for c in df.columns}

    not_null_cols = fsd_df[
        fsd_df["is_not_null"] == True
    ]["column_name"].str.strip().str.upper().tolist()

    for col_name in not_null_cols:
        # Skip columns absent from dataset (caught by schema check)
        if col_name not in dataset_cols_upper:
            continue

        actual_col_name = dataset_cols_upper[col_name]
        null_count      = df[actual_col_name].isna().sum()
        total_rows      = len(df)

        if null_count > 0:
            null_pct = round((null_count / total_rows) * 100, 2)
            anomalies.append({
                "check_type"       : "NULL",
                "column_name"      : col_name,
                "anomaly_type"     : "NULL_VIOLATION",
                "fsd_expected"     : "NOT NULL (Is Not Null? = Y)",
                "actual_value"     : f"{null_count} null(s) found out of {total_rows} rows ({null_pct}%)",
                "affected_row_count": null_count,
                "details"          : (
                    f"Column '{col_name}' is marked as NOT NULL in FSD but contains "
                    f"{null_count} null value(s) ({null_pct}% of {total_rows} total rows)."
                ),
            })

    return pd.DataFrame(anomalies)

# ============================================================
# SECTION 4: MASTER DQM RUNNER
# Orchestrates all checks and consolidates into one output
# ============================================================

In [ ]:
def run_dqm_pipeline(
    df: pd.DataFrame,
    fsd_df: pd.DataFrame,
    dataset_name: str = "MPATH_DTC_DISPLAY",
    channel: str = "DTC Display",
) -> pd.DataFrame:
    """
    Runs all DQM checks against the provided dataset and FSD schema.

    Parameters
    ----------
    df           : The actual dataset to validate (pandas DataFrame).
    fsd_df       : FSD schema DataFrame with columns:
                   ['column_name', 'fsd_dtype', 'is_not_null'].
    dataset_name : Name tag for the dataset (used in output metadata).
    channel      : Channel name (used in output metadata).

    Returns
    -------
    anomalies_df : Consolidated DataFrame of all anomalies found.
    """
    print("=" * 65)
    print(f"  DQM PIPELINE  |  Dataset: {dataset_name}  |  Channel: {channel}")
    print(f"  Run Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 65)

    # Normalize dataset column names to UPPER for consistent matching
    df = df.copy()
    df.columns = df.columns.str.strip().str.upper()

    all_anomalies = []

    # ── Run Check 1: Schema ──────────────────────────────────
    print("\n[1/3] Running Schema Check ...")
    schema_anomalies = check_schema(df, fsd_df)
    all_anomalies.append(schema_anomalies)
    print(f"      → {len(schema_anomalies)} anomaly(ies) found.")

    # ── Run Check 2: Data Types ──────────────────────────────
    print("\n[2/3] Running Data Type Check ...")
    dtype_anomalies = check_datatypes(df, fsd_df)
    all_anomalies.append(dtype_anomalies)
    print(f"      → {len(dtype_anomalies)} anomaly(ies) found.")

    # ── Run Check 3: Null Values ─────────────────────────────
    print("\n[3/3] Running Null Value Check ...")
    null_anomalies = check_nulls(df, fsd_df)
    all_anomalies.append(null_anomalies)
    print(f"      → {len(null_anomalies)} anomaly(ies) found.")

    # ── Consolidate All Anomalies ────────────────────────────
    anomalies_df = pd.concat(all_anomalies, ignore_index=True)

    # Add metadata columns
    anomalies_df.insert(0, "dataset_name", dataset_name)
    anomalies_df.insert(1, "channel",      channel)
    anomalies_df.insert(2, "run_timestamp", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

    # Enforce output column order
    output_cols = [
        "dataset_name",
        "channel",
        "run_timestamp",
        "check_type",
        "column_name",
        "anomaly_type",
        "fsd_expected",
        "actual_value",
        "affected_row_count",
        "details",
    ]
    anomalies_df = anomalies_df[output_cols]

    # ── Summary ──────────────────────────────────────────────
    total = len(anomalies_df)
    print("\n" + "-" * 65)
    print(f"  DQM COMPLETE  |  Total Anomalies Found: {total}")
    print("-" * 65)
    if total > 0:
        summary = anomalies_df.groupby("check_type")["anomaly_type"].count()
        for check, count in summary.items():
            print(f"    {check:<10} : {count} anomaly(ies)")
    print("=" * 65 + "\n")

    return anomalies_df


# ============================================================
# SECTION 5: ENTRY POINT
# ============================================================

In [ ]:
# ----------------------------------------------------------
# STEP 1: Load your actual dataset
# Replace this block with your actual data source
# e.g., pd.read_csv(...), pd.read_excel(...), Spark → toPandas(), etc.


if __name__ == "__main__":

    df = pd.read_csv("MPATH_DTC_DISPLAY.csv")
    #df = spark_df.toPandas()               # if reading from Spark/Databricks


In [ ]:
# ----------------------------------------------------------
# STEP 2: Run the DQM pipeline
# ----------------------------------------------------------

anomalies_df = run_dqm_pipeline(
        df           = df,
        fsd_df       = FSD_DF,
        dataset_name = "MPATH_DTC_DISPLAY",
        channel      = "DTC Display",
    )


In [ ]:
# ----------------------------------------------------------
# STEP 3: View & Export results
# ----------------------------------------------------------

print(anomalies_df.to_string(index=False))

# Export to Excel / CSV
# anomalies_df.to_excel("MPATH_DTC_DISPLAY_DQM_Report.xlsx", index=False)
# anomalies_df.to_csv("MPATH_DTC_DISPLAY_DQM_Report.csv",  index=False)
